# Demo 22 — Comparing model structures (`model_comparison`)

Which fixed-effect structure do the data support: the additive model, every
two-way interaction, or the full factorial? `options.model_comparison = True`
answers that next to the analysis, without changing it: `Summary.txt` gets a table
with the maximum-likelihood AIC and BIC of each structure (random effects held
fixed), the fitted one marked, and a verdict. Structures within 2 AIC of each other
are not told apart.

It is a **report, not a selection**. Choosing the structure by AIC and then quoting
that model's p-values makes them optimistic; the tested model follows from the
design, fixed before the data.

Part A uses `ToothGrowth` (two factors, supp × dose), Part B uses `npk` (three
factors, with a random intercept per block, as in Demo 9).

## Run on Google Colab

On [Google Colab](https://colab.research.google.com)? Run the cell below first —
it installs kbstatpy, its R packages, and the demo data (~3–5 min the first time).
It is a no-op when you run this notebook locally from the kbstatpy source tree.
Then run the cells below to see the tables and figures rendered inline.

In [ ]:
# Google Colab only: install kbstatpy + its R packages + the demo data.
# (Does nothing when the notebook runs locally from the source tree.)
import sys
if 'google.colab' in sys.modules:
    !curl -sSL https://raw.githubusercontent.com/kimbostroem/kbstatpy/master/demos/colab_setup.sh | bash

## Setup

In [ ]:
import os

from kbstatpy import Kbstat, KbstatOptions


def toothgrowth():
    o = KbstatOptions()
    o.in_file          = os.path.join(o.demo_dir, 'data/toothgrowth.csv')
    o.out_dir          = ''            # inline only; set a folder to also save
    o.y_units          = 'mm'
    o.x_order          = 'dose: low, medium, high'
    o.model_comparison = True
    return o


def npk():
    o = KbstatOptions()
    o.in_file          = os.path.join(o.demo_dir, 'data/npk.csv')
    o.out_dir          = ''
    o.y_units          = 'lb/plot'
    o.model_comparison = True
    return o


def show(kb):
    """The comparison table, as in Summary.txt."""
    mc = kb.model_comparison_table
    print(mc[['Structure', 'Fixed effects', 'npar', 'AIC', 'dAIC', 'fitted']]
          .to_string(index=False, float_format=lambda v: f'{v:.1f}'))

## A1. Two factors, defined as variables

`x = 'supp, dose'` fits the full factorial (`interaction = 'auto'`). The table
agrees: the additive model is 4.5 AIC worse.

In [ ]:
o = toothgrowth()
o.y = 'len'
o.x = 'supp, dose'
kb = Kbstat(o)
kb.run();
show(kb)

## A2. An additive formula

The fitted model is now `supp + dose`. The table names the full factorial as better
supported, and adds the caveat: switching to it is your decision, and p-values from
a structure chosen this way are optimistic.

In [ ]:
o = toothgrowth()
o.formula = 'len ~ supp + dose'
kb = Kbstat(o)
kb.run();
show(kb)

## A3. The same model, written differently

`supp + supp:dose` has no dose main effect in its text, but R codes it as dose
nested within supp: the same columns, the same fit as `supp * dose`. kbstatpy
matches a formula to the ladder by likelihood, not by its text, so the
full-factorial row is marked **fitted (same fit)**.

In [ ]:
o = toothgrowth()
o.formula = 'len ~ supp + supp:dose'
kb = Kbstat(o)
kb.run();
show(kb)

## B4. Three factors: all two-way interactions

With three factors the ladder has a middle rung, `(N + P + K)^2`: every two-way
interaction, no three-way. The formula is matched to it.

In [ ]:
o = npk()
o.formula = 'yield ~ (N + P + K)^2 + (1 | block)'
kb = Kbstat(o)
kb.run();
show(kb)

## B5. A partial interaction (Demo 9)

`N + P*K` is none of the rungs, so it gets its own row, **your formula**, and the
comparison still contains the model actually fitted.

In [ ]:
o = npk()
o.formula = 'yield ~ N + P*K + (1 | block)'
kb = Kbstat(o)
kb.run();
show(kb)

## Interpretation

- The table never changes the analysis: the ANOVA, post-hoc and plots belong to
  the model you specified.
- **ToothGrowth**: the interaction is supported (dAIC 4.5 against additive), in
  line with its significant `supp:dose` term.
- **npk**: all structures lie within about 2.6 AIC; the criteria do not separate
  them. That the data cannot tell these structures apart is a finding in itself,
  not a licence to pick the one with the nicest p-values.
- Matching is by likelihood, so formulas that R reads as the same model (`a + a:b`
  and `a * b`) land on the same row, whatever their spelling.